# NFPP Sodium-Ion BESS Performance Benchmarking and Latent Distribution Network State Estimation Using Network Realization Signatures

This notebook implements the complete research pipeline for the DFN-based optimization and the multi-feeder network state realization and anomaly detection framework.

In [ ]:
import os
import subprocess
import sys
from getpass import getpass

# Environment Setup
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('sodium-ion-ess'):
        get_ipython().system('git clone https://github.com/mhizterpaul/sodium-ion-ess.git')
        get_ipython().run_line_magic('cd', 'sodium-ion-ess')
    sys.path.append(os.getcwd())

# MP API Key configuration
if 'MP_API_KEY' not in os.environ:
    os.environ['MP_API_KEY'] = getpass("Enter Materials Project API Key: ")

!pip install pybamm numpy scipy matplotlib requests mp-api pymatgen pymoo mpi4py pint ufl OpenDSSDirect.py
!add-apt-repository -y ppa:fenics-packages/fenics
!apt update
!apt install -y fenicsx
import pybamm
import numpy as np
import matplotlib.pyplot as plt
print("Environment initialized.")

## Stage 2: Cell Optimization
Hierarchical Material Discovery + Structural Sensitivity Optimization.

In [ ]:
from src.cell_optimization.parameter_opts import HierarchicalOptimizer

print("Stage 2: Running Hierarchical Material & Structural Optimization...")
optimizer = HierarchicalOptimizer()
optimized_res = optimizer.run()

print("\n--- OPTIMIZATION RESULTS ---")
print("Optimized Design Variables per Objective:")
for obj, specs in optimized_res.get("opt_designs_per_objective", {}).items():
    print(f"\nObjective: {obj.capitalize()}")
    for k, v in specs.items():
        print(f"  {k:40s}: {v:12.6e}")

print("\nSelected Integrated Design Variables:")
for k, v in optimized_res.get("design_specs_representative", {}).items():
    print(f"  {k:40s}: {v:12.6e}")

print("\n--- OPTIMAL CANDIDATE: QM DATA & DERIVED CELL PARAMETERS ---")
mats = optimized_res.get("materials", {})
deltas = optimized_res.get("combined_deltas_representative", {})
for cat in ["cathode", "electrolyte"]:
    print(f"\n{cat.capitalize()} Material:")
    m_data = mats.get(cat, {})
    print(f"  Name: {m_data.get('name') or m_data.get('salt')}")
    print(f"  Formula: {m_data.get('formula')}")
    print("  QM/Physics Properties:")
    for pk, pv in m_data.get("properties", {}).items():
        print(f"    {pk:25s}: {pv}")

print("\nMapping to PyBaMM Parameter Deltas:")
for category, props in deltas.items():
    print(f"  [{category.upper()}]")
    for pk, pv in props.items():
        print(f"    {pk:45s}: {pv:+.4e}")

print("\n--- PERFORMANCE COMPARISON (OPTIMIZED CANDIDATE VS. NOMINAL) ---")
opt_p = optimized_res.get("metrics", {})

metrics_to_compare = [
    ("Energy [Wh]", "energy"),
    ("Power [W]", "power"),
    ("Stability Metric", "stability_metric"),
    ("Max Strain", "max_strain")
]

print(f"{'':40s} | {'Candidate Value':20s}")
print("-" * 65)
for label, key in metrics_to_compare:
    o_val = opt_p.get(key, 0.0)
    print(f"{label:40s} | {o_val:20.4e}")

## Stage 3: Stability Validation & Parameter Extraction
Performance evaluation and resistance profile generation for the digital twin.

In [ ]:
from src.cell_optimization.validate import OptimizationValidator

print("Stage 3: Running Stability Validation...")

# Map optimized design vector to parameter dict
design_specs = optimized_res.get("design_specs_representative", {})
deltas = optimized_res.get("combined_deltas_representative", {})

validator = OptimizationValidator(design_specs, deltas, engine=optimizer.engine)
results = validator.run_validation()

# Persist validation artifact for Stage 3.1 and Stage 4
import json
with open("final_validation.json", "w") as f:
    json.dump({"optimization": optimized_res, "validation": results}, f, indent=2)

print("\nStage 3.1: Running Parameter Extraction for Simscape...")
from src.simulation.tests import StabilityValidator
stab_validator = StabilityValidator()
envelope_res = stab_validator.validate_optimized_design()
stab_validator.export_to_json(envelope_res)

print("\n--- FULL MULTIPHYSICS SIMULATION RESULTS (BESS SCENARIOS) ---")
for k, v in envelope_res.items():
    if k in ["merged_params", "ssc_params"]: continue
    print(f"{k:40s}: {v}")

print("\nPerformance Metrics Summary:")
if results:
    for k, v in results.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

## Stage 4: Latent Distribution Network State Realization & Observability Analysis
In this stage, we simulate the 3-feeder distribution network under 15 operational scenarios using OpenDSS and QSTS simulations with SVD-based observability analysis of the network response Jacobian. For each scenario, we programmatically construct a randomized downstream network graph of 20-80 buses with mixed loads, capacitors, motors, switches, and topology reconfigurations (radial vs ring/loop). We extract and tabulate both steady-state feeder/transformer parameters and sub-cycle transient parameters, exporting the complete dataset to a CSV file.

In [ ]:
from src.power_plant.dataset import generate_realization_dataset

print("Stage 4.1: Executing Programmatic OpenDSS QSTS Perturbed Scenarios...")
scenario_data = generate_realization_dataset(n_scenarios=15)
print("\nScenarios executed successfully. Results exported to 'src/simulation/scenario_results.csv'.")

In [ ]:
import pandas as pd
from IPython.display import display, HTML

print("Stage 4.2: Tabulation of Response Jacobian SVD Parameters")
df = pd.read_csv("src/simulation/scenario_results.csv")

# Select and format the Response Jacobian SVD and conditioning parameters
svd_cols = [
    "scenario_index", "topology_type", "simulated_event", "active_feeder",
    "jacobian_sigma_1", "jacobian_sigma_2", "jacobian_sigma_3", "jacobian_sigma_4",
    "observability_condition_kappa"
]
df_svd = df[svd_cols].copy()
df_svd.columns = [
    "Scenario", "Topology", "Simulated Event", "Active Feeder",
    "sigma_1", "sigma_2", "sigma_3", "sigma_4", "Condition Number (kappa)"
]

# Display as a structured HTML table
display(HTML("<h3>Response Jacobian SVD Parameters across Simulated Scenarios</h3>"))
display(df_svd)

In [ ]:
print("Stage 4.3: Tabulation of Feeder and Transformer Parameters")

# Select and format the steady-state boundary measurements and transformer metrics
feeder_cols = [
    "scenario_index", "topology_type", "simulated_event",
    "feeder1_voltage_mag_kv", "feeder1_voltage_unbalance_pct",
    "feeder1_current_mag_amp", "feeder1_current_unbalance_pct",
    "feeder1_pf", "feeder1_eq_impedance_ohm",
    "transformer1_hv_voltage_v", "transformer1_hv_current_amp",
    "transformer1_loading_pct", "transformer1_copper_loss_kw",
    "transformer1_core_loss_kw", "transformer1_voltage_regulation_pct",
    "transformer1_eq_impedance_ohm", "transformer1_tap_position"
]
df_feeder = df[feeder_cols].copy()
df_feeder.columns = [
    "Scenario", "Topology", "Event",
    "F1 V (kV)", "F1 V Unbalance (%)",
    "F1 I (A)", "F1 I Unbalance (%)",
    "F1 PF", "F1 Z_eq (Ohm)",
    "T1 HV V (V)", "T1 HV I (A)",
    "T1 Loading (%)", "T1 Cu Loss (kW)",
    "T1 Core Loss (kW)", "T1 Regulation (%)",
    "T1 Z_eq (Ohm)", "T1 Tap Pos"
]

# Display as a structured HTML table
display(HTML("<h3>Feeder and Transformer Parameters under Operational Conditions</h3>"))
display(df_feeder)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

print("Stage 4.4: Analytical Rendering & Observability Interpretation")
df = pd.read_csv("src/simulation/scenario_results.csv")

# Plot singular value spectra and perturbative sensitivities
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scenarios_to_plot = [0, 4, 11]
colors = ['r', 'g', 'b']
for idx, col in zip(scenarios_to_plot, colors):
    row = df.iloc[idx]
    sigmas = [row["jacobian_sigma_1"], row["jacobian_sigma_2"], row["jacobian_sigma_3"], row["jacobian_sigma_4"]]
    axes[0].plot(range(1, 5), sigmas, marker='o', color=col, label=f"Scenario {idx} ({row['simulated_event'].replace('_', ' ').title()})")
axes[0].set_xlabel("Singular Value Index")
axes[0].set_ylabel("Singular Value Magnitude")
axes[0].set_title("Response Jacobian Singular Value Spectra")
axes[0].set_xticks(range(1, 5))
axes[0].grid(True)
axes[0].legend()

axes[1].bar(df["scenario_index"] - 0.2, df["feeder1_dv_dp"], width=0.4, color='orange', label="dV/dP (Feeder 1)")
axes[1].bar(df["scenario_index"] + 0.2, df["feeder1_dv_dq"], width=0.4, color='teal', label="dV/dQ (Feeder 1)")
axes[1].set_xlabel("Scenario Index")
axes[1].set_ylabel("Voltage Sensitivity")
axes[1].set_title("Feeder 1 Voltage Sensitivities to Load Perturbations")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n--- Key Scientific Observability Observations ---")
print("1. The Jacobian singular value spectra show distinct decay patterns that reveal the observable directions of the network response.")
print("2. Complex equivalent impedance Z_eff and direct voltage sensitivities dV/dP and dV/dQ provide robust, mathematically grounded signatures.")
print("3. Removing synthetic wave emulations in favor of true physical load perturbations enables defensible network realization.")